# Calibration

In [ ]:
"""
Mainboard for the baseline example of the BASEforHANK package, calibration.
"""

using PrettyTables, Printf;

## ------------------------------------------------------------------------------------------
## Header: set up paths, pre-process user inputs, load module
## ------------------------------------------------------------------------------------------

root_dir = replace(Base.current_project(), "Project.toml" => "");
cd(root_dir);

# set up paths for the project
paths = Dict(
    "root" => root_dir,
    "src" => joinpath(root_dir, "src"),
    "bld" => joinpath(root_dir, "bld"),
    "src_example" => @__DIR__,
    "bld_example" => replace(@__DIR__, "examples" => "bld") * "_calib",
);

# create bld directory for the current example
mkpath(paths["bld_example"]);

# pre-process user inputs for model setup
include(paths["src"] * "/Preprocessor/PreprocessInputs.jl");
include(paths["src"] * "/BASEforHANK.jl");
using .BASEforHANK;

# set BLAS threads to the number of Julia threads, prevents grabbing all
BASEforHANK.LinearAlgebra.BLAS.set_num_threads(Threads.nthreads());

## ------------------------------------------------------------------------------------------
## Initialize: set up model parameters
## ------------------------------------------------------------------------------------------

m_par = ModelParameters();

One way users can bring the model to the data is through calibration. The package provides users this possibility, by providing a main board, `main_calibration.jl`, to run their calibration of choice. Calibration here will mean to minimize the distance between specified target moments, e.g., the capital-to-ouput ratio, from the data and their model moment counterpart.

To run your own calibration, you would simply change 1 function from `main_calibration.jl`. The file will be identical to `main.jl` in that it calls all the necessary inputs to run what you would like, so it will look familiar. 

Below, we describe the functions within the file necessary to run a calibration and explain step-by-step how to run your own calibration. The notebook also provides several examples, which can be used off-the-shelf if you'd like. The beauty in the set-up is the user only needs to define one function, which will be a block of equations, which is needed to compute the model-moments.

It's important to note that the current calibration set-up works for any package model.

## Calibration function

~~~
m_par = BASEforHANK.SteadyState.run_calibration(moments_function, cal_dict, m_par; solver="NelderMead")
~~~

Here, we present the top-level function `run_calibration` that runs the calibration. To run `run_calibration`, you need to pass the following arguments: 

- `moments_function::Function` **will be the only function that the user needs to write and define**. The function will need to generate aggregates, which will be needed to generate the model-moments. We will provide three examples below so the user has a sense of how to precisely do this. Although the user will have to find the equations to generate the aggregates they wish to match, it buys them the flexibillity of being able to stipulate any moment (as long as there is a model-counterpart). 

- `m_par` will be an instance of `::ModelParameters` and its entries are documented elsewhere in the documentation. The optimizer will use its entries to define the initial parameter vector for the initialization of the optimization (depending on the optimizer) and then after running the optimization, the function will output a new set of parameters, all contained in `m_par`.
 
- `solver::String` is an option for the optimizer of choice, for which we provide one local optimizer: `"NelderMead"` and one global optimizer: `"BBO"`. Details on the solvers can be found within their respective packages. The default solver is `"NelderMead"`.

- `cal_dict::Dict()` is short for calibration dictionary and it will contain a fixed set of keys needed for the optimization. `cal_dict` has two required keys and 1 optional key, as shown below: 

In [ ]:
using Optim;

# Initialize the calibration dictionary
cal_dict = Dict();
cal_dict["target_moments"] = Dict();

# Required (entries for now are for demonstration purposes)
cal_dict["params_to_calibrate"] = [:β, :λ];

cal_dict["target_moments"]["B/K"] = 0.25;  # add the Liquid to illiquid ratio as target

# Optional, as we toss in default options if the user specifies nothing
cal_dict["opt_options"] = Optim.Options(;
    time_limit=1200,
    f_reltol=1e-3,
    store_trace=true,
    show_trace=true,
    show_every=30,
);

The options for `"NelderMead"` are just `Optim.Options`. The user can specify whichever options they please and they are passed to the solver. For `"BBO"`, just specify the options using a tuple. An example is provided below, in Example 3.

As mentioned in the comments above, the package does provide default options for the calibration, but it is highly encouraged you understand these options and specify them according to your setting. 

For the remainder of the notebook, focus will be on how to define the `moments_function`, as this will be the key component for the calibration.

## Example 1: Defining a `moments_function` with One Moment

In [ ]:
# Initialize the calibration dictionary
cal_dict_one_target = Dict();
cal_dict_one_target["target_moments"] = Dict();

# Required (entries for now are for demonstration purposes)
cal_dict_one_target["params_to_calibrate"] = [:β];
cal_dict_one_target["target_moments"]["K/Y"] = 11.22 / 4;  # Capital to output ratio

cal_dict_one_target["opt_options"] = Optim.Options(;
    time_limit=1200,
);

In [ ]:
# `moment_function` example
function calculate_one_moment(m_par)
    # steady state at the prior mode
    ss_full = quiet_call(call_find_steadystate, m_par)

    # extract numerical parameters for the calculation of the aggregates
    n_par = ss_full.n_par

    # compute aggregates
    args_hh_prob = BASEforHANK.IncomesETC.compute_args_hh_prob_ss(ss_full.KSS, m_par, n_par)
    BASEforHANK.Parsing.@read_args_hh_prob() # Gives us a few aggregates, see 'args_hh_prob_names'

    # Calculate aggregates
    K = ss_full.KSS
    Y = BASEforHANK.IncomesETC.output(m_par.Z, K, N, m_par)

    # Calculate model moments
    model_moments = Dict("K/Y" => K / Y / 4.0)

    return model_moments
end;

An example of a `moments_function` that the user will need to write can be seen above. It's called `calculate_one_moment::Function` and its to demonstrate a simple case with one target. The function does the following:

 (1) **Computing some aggregates:** `call_find_steadystate(m_par)` is ran. This will always need to be called, since this returns certain steady-state aggregates necessary in its own right and for the computation of other aggregates. On top, `BASEforHANK.IncomesETC.compute_args_hh_prob_ss` will always need to be called, since this returns additional aggregates possibly relevant for the user's calibration. From this, the user would be able to compute plenty moments of interest. 

(2) **Computing the model-moments:** The model-moments of interest to the user will need to be defined by in a dictionary `model_moments::Dict()`. Here in this example, the user wanted the capital-to-output ratio and so, the user needs to find its counterpart in the code. In this case, capital in the package code is `K` and output is `Y`. The capital-to-output ratio is then computed as `K/Y`. The actual key in the dictionary, denoted as a `String`, `"K/Y"`, can be anything and will only appear in the end, when we present the final model-moments generated from the parameters returned by the optimizer. For example, the user can define it as: `model_moments = Dict("Capital-to-Output" => K / Y / 4.0)`. BUT: the keys for the moments within `model_moments` has to match the keys of `cal_dict` e.g., `cal_dict["target_moments"]["Capital-to-Output"] = K / Y / 4.0)`. Note: the division by 4 is to have capital (a stock variable) relative to quarterly output (output divided by 4).

The function in the end returns the model-moments, which the optimizer then compares to the target during the optimization procedure and ultimately, in a `PrettyTable`, once the optimizer has ran to the specified criteria.

With the arguments defined above, we can now try running a calibration:

Below, we present two examples where this function may require a few additional lines of code to compute aggregates necessary to compute the model-moments.

In [ ]:
# Run calibration. Exports parameters
m_par = BASEforHANK.SteadyState.run_calibration(
    calculate_one_moment,
    cal_dict_one_target,
    m_par;
    solver="NelderMead",
);

# Example 2: Defining a Moments Function with Two Moments

In [ ]:
# Initialize the calibration dictionary
cal_dict_two_targets = Dict();
cal_dict_two_targets["target_moments"] = Dict();

# Parameters of choice
cal_dict_two_targets["params_to_calibrate"] = [:β, :λ];

# Target moments of choice (with quarterly flow variables)
cal_dict_two_targets["target_moments"]["K/Y"] = 11.22 / 4;
cal_dict_two_targets["target_moments"]["B/K"] = 0.25;

cal_dict_two_targets["opt_options"] = Optim.Options(
    time_limit=1200,
    f_reltol=1e-3,
);

In [ ]:
function calculate_two_moments(m_par)
    # steady state at the prior mode
    ss_full = quiet_call(call_find_steadystate, m_par)

    # extract numerical parameters for the calculation of the aggregates
    n_par = ss_full.n_par

    # Compute aggregates
    args_hh_prob = BASEforHANK.IncomesETC.compute_args_hh_prob_ss(ss_full.KSS, m_par, n_par)
    BASEforHANK.Parsing.@read_args_hh_prob() # Gives us a few aggregates e.g., N ..., see 'args_hh_prob_names' for more

    # Compute aggregates
    K = ss_full.KSS
    Y = BASEforHANK.IncomesETC.output(m_par.Z, K, N, m_par)
    B = sum(ss_full.distrSS .* ss_full.n_par.mesh_b)

    # Calculate model moments
    model_moments = Dict("K/Y" => K / Y / 4)
    model_moments["B/K"] = B / K

    return model_moments
end;

Here is another example of the `moments_function`. The user, as you see, has stipulated to target the capital-to-output ratio, `"K/Y"`, as before, but now includes the bonds-to-capital ratio `"B/K"`. 

Different to the first example, the user needs to compute `B`, which is the code counterpart of bonds. To compute `B`, the user would need to access `ss_full`, to use `distrSS`. `distrSS` is the steady-state distribution of idiosyncratic states and can be thought of as the mass of individuals. To then compute total liquid assets, you can multiply the mass with `mesh_b`, which is the same size of `distrSS`, containing bond levels across grid points. 

Finally, similar to above, we create the `model_moments` dictionary, this time, augmented with the new moment.

Again, the user only needs to define this function and in particular, "plug-in" the lines that pertain to `# Compute aggregates` and `# Calculate model moments`. Note that division by 4 can be omitted here as the new target for the capital output ratio already uses quarterly GDP analogous to the model.

In [ ]:
# Run calibration. Exports parameters
m_par = BASEforHANK.SteadyState.run_calibration(
    calculate_two_moments,
    cal_dict_two_targets,
    m_par;
    solver="NelderMead",
);

# Example 3: Moments Function with Five Moments

In [ ]:
# Initialize the calibration dictionary
# For BBO
cal_dict_BBO = Dict(
    "params_to_calibrate" => [:β, :λ, :Tlev, :ζ, :Rbar],
    "target_moments" => Dict( # User-defined targets # these are from paper
        "K/Y" => 11.22 / 4,  # Capital-output (quarterly) ratio
        "B/K" => 0.25,  # Liquid to illiquid ratio
        "G/Y" => 0.20,  # Gov. spending-output (annualy) ratio
        "T10W" => 0.67,  # Top 10% wealth share
        "Frac Borrowers" => 0.16,  # Fraction of borrowers
    ),
    # One must change options for their respective setting!
    "opt_options" => (
        SearchRange=[
            (0.90, 0.999), # β
            (0.01, 0.2), # λ
            (1.0, 1.5), # Tlev
            (0.0, 0.0005), # ζ
            (0.0, 0.05), # Rbar
        ],
        Method=:adaptive_de_rand_1_bin_radiuslimited,
        MaxTime=10800, # 3 hours
        TraceInterval=30,
        TraceMode=:compact,
        TargetFitness=1e-3,   # stops if fitness ≤ tolerance
    ),
);

# For Nelder-Mead
cal_dict_NM = Dict(
    "params_to_calibrate" => [:β, :λ, :Tlev, :ζ, :Rbar],
    "target_moments" => Dict( # User-defined targets # these are from paper
        "K/Y" => 11.22 / 4,  # Capital-output ratio
        "B/K" => 0.25,  # Liquid to illiquid ratio
        "G/Y" => 0.20,  # Gov. spending-output ratio
        "T10W" => 0.67,  # Top 10% wealth share
        "Frac Borrowers" => 0.16,  # Fraction of borrowers
    ),
    # One must change options for their respective setting!
    "opt_options" => Optim.Options(;
        time_limit=10800,
        show_trace=true,
        show_every=10, # iteration count ... an iteration takes about a minute
        f_reltol=1e-3,   # stops if fitness ≤ tolerance
    ),
);

In [ ]:
function calculate_moments(m_par)
    # calculate the steady state associated with the current parameter
    ss_full = quiet_call(call_find_steadystate, m_par)

    # extract numerical parameters for the calculation of the aggregates
    n_par = ss_full.n_par

    # Compute aggregates
    args_hh_prob = BASEforHANK.IncomesETC.compute_args_hh_prob_ss(ss_full.KSS, m_par, n_par)
    BASEforHANK.Parsing.@read_args_hh_prob()

    # Compute aggregates
    # capital
    K = ss_full.KSS

    # bonds
    B = sum(ss_full.distrSS .* ss_full.n_par.mesh_b)

    # marginal cost, wages, output
    mc = 1.0 ./ m_par.μ
    wF = BASEforHANK.IncomesETC.wage(mc, m_par.Z, K, N, m_par)
    Y = BASEforHANK.IncomesETC.output(m_par.Z, K, N, m_par)

    # taxes
    T =
        (Tbar - 1.0) .* (1.0 ./ m_par.μw .* wF .* N) + # labor income
        (Tbar - 1.0) .* Π_E + # profit income
        (Tbar - 1.0) * ((1.0 .- 1.0 ./ m_par.μw) .* wF .* N) # union profit income

    # government bonds
    RRL = m_par.RRB # in ss
    qΠ = m_par.ωΠ .* (1.0 .- 1.0 ./ m_par.μ) .* Y ./ (RRL .- 1 .+ m_par.ιΠ) + 1.0

    # qΠ == 1.0 ? (log(RBPrime / πPrime)) :
    # (log(RBPrime / πPrime)) -
    # (log(((qΠPrime - 1.0) * (1 - ιΠ) + ωΠ * Π_FPrime) / (qΠ - 1.0)))

    Bgov = B .- qΠ .+ 1.0

    # government spending
    G = T - (RRL - 1.0) * Bgov

    # calculate the fraction of borrowers
    distrSS = ss_full.distrSS
    fr_borr = BASEforHANK.eval_cdf(sr_full.distrSS, :b, sr_full.n_par, 0.0);

    # Price of capital is 1 in the steady-state
    q = 1

    # calculate the Top 10% wealth share
    total_wealth = Array{eltype(distrSS)}(undef, n_par.nk .* n_par.nb)
    for k = 1:(n_par.nk)
        for b = 1:(n_par.nb)
            total_wealth[b+(k-1)*n_par.nb] = n_par.grid_b[b] .+ q .* n_par.grid_k[k]
        end
    end
    # Wealth shares
    IX = sortperm(total_wealth)
    total_wealth = total_wealth[IX]
    total_wealth_pdf = sum(distrSS; dims=3)
    total_wealth_pdf = total_wealth_pdf[IX]
    total_wealth_cdf = cumsum(total_wealth_pdf)
    total_wealth_w = total_wealth .* total_wealth_pdf # weighted
    wealthshares = cumsum(total_wealth_w) ./ sum(total_wealth_w)
    TOP10Wshare =
        1.0 -
        BASEforHANK.Tools.mylinearinterpolate(total_wealth_cdf, wealthshares, [0.9])[1]

    # Compute model moments
    model_moments = Dict(
        "K/Y" => K / Y / 4.0,
        "B/K" => B / K,
        "G/Y" => G / Y,
        "T10W" => TOP10Wshare,
        "Frac Borrowers" => fr_borr,
    )
    return model_moments
end;

The last example here is very extensive, but shows the degree of flexibility offered by the calibration set-up. Here, we define five model targets and calibrate five parameters in the model to match them. This is exact pairing of parameter with target is from Bayer et. al. Shocks, Frictions, and Inequality.

In this case, where the dimensionality is a bit higher, the calibration is done using an Adaptive Differential-Evolution optimizer from `BlackBoxOptim`. You can look up the options for their optimizers on their page. 

After figuring out which options you would like, you can pass them as a tuple.

The optimizer in this case **requires** the user to stipulate bounds on the parameters, so the optimizer knows where to explore. You can define them inside of `cal_dict` under `opt_options` -> `SearchRange`. You need to specify the bounds as a `Vector` of tuples, where the first entry is the lower bound and the second entry is the upper bound. **This is only required for the solver "BBO"**. The more generous the bounds, the longer the optimization time you must define.

For this example, to calculate the five moments, we need several aggregates: `K`, `Y`, `B`, `G`, `T`, `BGov`, `T10W`, `distr_b`, and `Frac_Borrowers`. You can see above how we compute these. The constraints placed on the optimizer will define the time it takes to run. 

**The point is:** the user will have to compute these aggregates here. Once this is done, everything is done under the hood and one can proceed with the optimized parameters, which are stored in the updated `m_par`.  


**Tips**: 
- The optimizer is quite good, so test perhaps with 20 minutes to see how close it gets. For a lower tolerance, more time (e.g., hours) may be necessary. 

- If you run into trouble, try different (less generous bounds) bounds. 



In [ ]:
# Run calibration. Exports parameters
m_par = BASEforHANK.SteadyState.run_calibration(
    calculate_moments,
    cal_dict_NM,
    m_par;
    solver="NelderMead",
);